In [1]:
from datasets import Dataset

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import re


c:\Users\agsto\Desktop\CMPE-252-summary-project\summary_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())       # True if GPU is usable
print("CUDA version:", torch.version.cuda)                # CUDA version PyTorch was built with
print("Number of GPUs:", torch.cuda.device_count())       # Number of GPUs detected
if torch.cuda.is_available():
    print("GPU name:", torch.cuda.get_device_name(0))

PyTorch version: 2.10.0+cu126
CUDA available: True
CUDA version: 12.6
Number of GPUs: 1
GPU name: NVIDIA GeForce RTX 4070 SUPER


## Importing and cleaning data set

In [5]:
section_path = "section_train_0.parquet" #file path
section = pd.read_parquet(section_path)

In [6]:
documention_path_0 = "document_train_0.parquet" #file path
document_0 = pd.read_parquet(documention_path_0)

documention_path_1 = "document_train_1.parquet"
document_1 = pd.read_parquet(documention_path_1)

documention_path_2 = "document_train_2.parquet"
document_2 = pd.read_parquet(documention_path_2)

documention_path_3 = "document_train_3.parquet"
document_3 = pd.read_parquet(documention_path_3)

In [7]:
document = pd.concat([document_0, document_1, document_2, document_3])

In [8]:
document['abstract'] = document['abstract'].str.replace(' \n', '', regex=False) #remove \n
document['abstract'] = document['abstract'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
document['abstract'] = document['abstract'].str.replace('  ', ' ', regex=False) #replace double space with single space

document['article'] = document['article'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
document['article'] = document['article'].str.replace('  ', ' ', regex=False) #replace double space with single space

## Remove samples with missing info and samples with too much noise

In [9]:
article_summary = document['article'].str.len().describe()
article_summary

count     54144.000000
mean      30196.147292
std       22630.985389
min           0.000000
25%       16117.500000
50%       24903.000000
75%       38290.500000
max      607645.000000
Name: article, dtype: float64

In [10]:
abstract_summary = document['abstract'].str.len().describe()
abstract_summary

count    54144.000000
mean      1490.463542
std       2872.869021
min          7.000000
25%        659.000000
50%        938.000000
75%       1324.000000
max      69493.000000
Name: abstract, dtype: float64

In [11]:
document = document[document['article'].str.len() >= article_summary['25%']] 
document = document[document['article'].str.len() <= article_summary['75%']]  

document = document[document['abstract'].str.len() >= abstract_summary['25%']]
document = document[document['abstract'].str.len() <= abstract_summary['75%']]

document = document.drop_duplicates()
document = document.reset_index(drop=True)

## BART (No training yet)

In [15]:
from transformers import BartTokenizer, BartForConditionalGeneration
import torch

model_name = "facebook/bart-large-cnn"

tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name, output_attentions=True)

model.config.output_attentions = True
model.config.return_dict = True

model.eval()

Loading weights: 100%|██████████| 511/511 [00:00<00:00, 1148.34it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   


BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50264, 1024, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=True)
        

In [16]:
text = document['article'][0]

inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)

generated = model.generate(
    inputs["input_ids"],
    num_beams=4,
    max_length=500,
    min_length=100,
    #length_penalty=2.0, # can change min and max length as well as this to see if we want to have the summary be more brief     
    early_stopping=True,
    no_repeat_ngram_size=3,
    do_sample=False,
    return_dict_in_generate=True
)

generated_ids = generated.sequences

summary = tokenizer.decode(generated_ids[0], skip_special_tokens=True)
print(summary)

#for attention values
outputs = model(
    input_ids=inputs["input_ids"],
    decoder_input_ids=generated_ids,
    output_attentions=True,
    return_dict=True
)

cross_attentions = outputs.cross_attentions

additive models provide an important family of models for semiparametric regression or classification. Some reasons for the success of additive models are their increased flexibility when compared to linear or generalized linear models and their increased interpretability. In this paper we address the open question whether an svm with an additive kernel can provide a substantially better learning rate in high dimensions. We will not address the question how to check whether the assumption of an additive model is satisfied because this would be a topic of a paper of its own.


In [17]:
print(len(cross_attentions))
print(cross_attentions[0].shape)

12
torch.Size([1, 16, 105, 1024])


In [18]:
cross_attentions

(tensor([[[[1.9530e-01, 6.6491e-07, 8.6813e-10,  ..., 4.6027e-08,
            1.8740e-08, 3.3442e-02],
           [7.8566e-02, 3.6558e-05, 1.2620e-08,  ..., 3.2075e-08,
            1.8732e-09, 4.3937e-02],
           [1.4969e-03, 9.6271e-01, 2.0434e-06,  ..., 9.9358e-10,
            1.5781e-08, 1.2946e-03],
           ...,
           [2.9099e-03, 1.7626e-07, 4.7626e-07,  ..., 3.8807e-09,
            3.5607e-07, 9.4704e-04],
           [1.6940e-01, 9.5440e-07, 6.9736e-07,  ..., 5.7952e-09,
            1.4324e-08, 6.0313e-02],
           [7.8984e-02, 1.0426e-08, 5.9409e-11,  ..., 3.7038e-09,
            1.4941e-07, 8.5193e-02]],
 
          [[7.5060e-02, 1.1381e-04, 1.7044e-04,  ..., 1.7028e-04,
            1.5922e-04, 3.0086e-02],
           [8.7623e-02, 6.6379e-04, 3.1574e-04,  ..., 4.5471e-05,
            4.4424e-05, 2.2373e-02],
           [2.5617e-01, 1.4498e-03, 4.5393e-04,  ..., 5.4525e-05,
            8.1151e-05, 2.6895e-02],
           ...,
           [2.5342e-01, 7.4780e-04, 4.

## Validation and Test cleaning

In [19]:
model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name, output_attentions=True, torch_dtype = torch.bfloat16)

Loading weights: 100%|██████████| 511/511 [00:00<00:00, 980.83it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]    


In [20]:
validation_path = "document_validation.parquet" #file path
validation = pd.read_parquet(validation_path)

test_path = "document_test.parquet"
test = pd.read_parquet(test_path)

In [21]:
validation['abstract'] = validation['abstract'].str.replace(' \n', '', regex=False) #remove \n
validation['abstract'] = validation['abstract'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
validation['abstract'] = validation['abstract'].str.replace('  ', ' ', regex=False)

validation['article'] = validation['article'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
validation['article'] = validation['article'].str.replace('  ', ' ', regex=False) #replace double space with single space

In [22]:
validation_article_summary = validation['article'].str.len().describe()

validation_article_summary

count      6436.000000
mean      30036.521287
std       19167.893171
min        1028.000000
25%       17368.500000
50%       25678.500000
75%       37804.500000
max      206131.000000
Name: article, dtype: float64

In [23]:
validation_abstract_summary = validation['abstract'].str.len().describe()

validation_abstract_summary

count    6436.000000
mean      929.612803
std       330.412665
min       238.000000
25%       673.750000
50%       905.500000
75%      1172.000000
max      1863.000000
Name: abstract, dtype: float64

In [24]:
validation = validation[validation['article'].str.len() >= validation_article_summary['25%']] 
validation = validation[validation['article'].str.len() <= validation_article_summary['75%']]  

validation = validation[validation['abstract'].str.len() >= validation_abstract_summary['25%']]
validation = validation[validation['abstract'].str.len() <= validation_abstract_summary['75%']]

validation = validation.drop_duplicates()
validation = validation.reset_index(drop=True)

In [25]:
test['abstract'] = test['abstract'].str.replace(' \n', '', regex=False) #remove \n
test['abstract'] = test['abstract'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
test['abstract'] = test['abstract'].str.replace('  ', ' ', regex=False)

test['article'] = test['article'].str.replace(r'@\w+', '', regex=True) # remove @ formulas
test['article'] = test['article'].str.replace('  ', ' ', regex=False) #replace double space with single space

In [26]:
test_article_summary = test['article'].str.len().describe()

test_article_summary

count      6440.000000
mean      30083.701242
std       19607.036626
min         599.000000
25%       17539.750000
50%       25769.000000
75%       38066.000000
max      454025.000000
Name: article, dtype: float64

In [27]:
test_abstract_summary = test['abstract'].str.len().describe()

test_abstract_summary

count    6440.000000
mean      936.243634
std       323.882370
min       203.000000
25%       686.000000
50%       918.500000
75%      1177.250000
max      1814.000000
Name: abstract, dtype: float64

In [28]:
test = test[test['article'].str.len() >= test_article_summary['25%']] 
test = test[test['article'].str.len() <= test_article_summary['75%']]  

test = test[test['abstract'].str.len() >= test_abstract_summary['25%']]
test = test[test['abstract'].str.len() <= test_abstract_summary['75%']]

test = test.drop_duplicates()
test = test.reset_index(drop=True)

## Training

In [29]:
train_dataset = Dataset.from_pandas(document)
validation_dataset   = Dataset.from_pandas(validation)
test_dataset  = Dataset.from_pandas(test)

In [ ]:
label = tokenizer(
    list(train_dataset["abstract"]),
    truncation=True,
    max_length=1024,
    padding=False
)

# Get lengths
label_lengths = [len(ids) for ids in label["input_ids"]]

In [31]:
abstract_tokens_summary = pd.DataFrame(label_lengths).describe()[0]
abstract_tokens_summary

count    15490.000000
mean       193.410781
std         42.827134
min        106.000000
25%        159.000000
50%        189.000000
75%        223.000000
max        544.000000
Name: 0, dtype: float64

In [32]:
token_len_IQR =  abstract_tokens_summary['75%'] - abstract_tokens_summary['25%']
1.5 * token_len_IQR

np.float64(96.0)

In [33]:
abstract_tokens_summary['25%'] - (1.5 * token_len_IQR)

np.float64(63.0)

In [34]:
abstract_tokens_summary['75%'] + (1.5 * token_len_IQR)

#pad to 320 should be a clean number

np.float64(319.0)

In [35]:
import nltk
import evaluate


nltk.download('punkt', quiet=True)
metric = evaluate.load('rouge')

In [30]:
def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # decode preds and labels
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    # rougeLSum expects newline after each sentence
    decoded_preds = ["\n".join(nltk.sent_tokenize(pred.strip())) for pred in decoded_preds]
    decoded_labels = ["\n".join(nltk.sent_tokenize(label.strip())) for label in decoded_labels]

    result = metric.compute(predictions=decoded_preds, references=decoded_labels, use_stemmer=True)
    return result

In [36]:
model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name, output_attentions=False, torch_dtype = torch.bfloat16).cuda()

Loading weights: 100%|██████████| 511/511 [00:00<00:00, 978.95it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   


In [37]:
def tokenize_document(dataset):
    model_input = tokenizer(dataset["article"], max_length = 1024, truncation = True)
    labels = tokenizer(dataset["abstract"], max_length = 320, truncation = True)

    model_input["labels"] = labels['input_ids']
    return model_input

In [38]:
tokenized_dataset = train_dataset.map(tokenize_document, batched=True)

Map: 100%|██████████| 15490/15490 [01:23<00:00, 185.69 examples/s]


In [39]:
from transformers import (
    BartTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    DataCollatorForSeq2Seq
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding="longest"  # pads to the longest sequence in each batch
)

In [40]:
validation_tokenized = validation_dataset.map(tokenize_document, batched=True, remove_columns=validation_dataset.column_names)
test_tokenized = test_dataset.map(tokenize_document, batched=True, remove_columns=test_dataset.column_names)

Map: 100%|██████████| 1715/1715 [00:09<00:00, 176.26 examples/s]


In [41]:
model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name, torch_dtype = torch.bfloat16).cuda()


Loading weights: 100%|██████████| 511/511 [00:00<00:00, 1003.95it/s, Materializing param=model.encoder.layers.11.self_attn_layer_norm.weight]   


In [47]:
from transformers import Seq2SeqTrainingArguments

training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_finetuned",      # folder to save checkpoints
    num_train_epochs=2,                  # number of passes through the dataset
    per_device_train_batch_size=4,       # safe for 12GB VRAM
    per_device_eval_batch_size=4,        # same as train batch size
    learning_rate=2e-5,                  # learning rate
    weight_decay=0.01,                   # regularization
    save_total_limit=3,                   # keep last 3 checkpoints
    eval_strategy="epoch",          # evaluate once per epoch
    save_strategy="epoch",                # save checkpoint once per epoch
    logging_strategy="steps",             # log every N steps
    logging_steps=100,                    # adjust if needed
    bf16=True,                            # mixed precision for VRAM efficiency
    predict_with_generate=True            # necessary for seq2seq evaluation
)

In [43]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=validation_tokenized,      
    processing_class=tokenizer,
    data_collator=data_collator
)

In [83]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,2.795941,2.686332
2,2.850331,2.683255


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.25it/s]


TrainOutput(global_step=7746, training_loss=2.7847567656044503, metrics={'train_runtime': 2220.0172, 'train_samples_per_second': 13.955, 'train_steps_per_second': 3.489, 'total_flos': 6.778703195406336e+16, 'train_loss': 2.7847567656044503, 'epoch': 2.0})

In [53]:
from transformers import BartForConditionalGeneration, BartTokenizer, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import load_dataset

# Re-load model, tokenizer, dataset, and training arguments
model = BartForConditionalGeneration.from_pretrained(r"bart_finetuned\checkpoint-7746").cuda()
tokenizer = BartTokenizer.from_pretrained(r"bart_finetuned\checkpoint-7746")

Loading weights: 100%|██████████| 512/512 [00:00<00:00, 1230.80it/s, Materializing param=model.shared.weight]                                   


In [ ]:
from transformers import Seq2SeqTrainer

# Set training args — keep num_train_epochs the same or higher than previous total
training_args = Seq2SeqTrainingArguments(
    output_dir="./bart_finetuned",
    num_train_epochs=3,  # total epochs, not additional
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    learning_rate=2e-5,
    weight_decay=0.01,
    save_total_limit=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    bf16=True,
    predict_with_generate=True
)

In [57]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=validation_tokenized,
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train(resume_from_checkpoint="./bart_finetuned/checkpoint-7746")

There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Epoch,Training Loss,Validation Loss
3,2.777663,2.682559


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.13it/s]


TrainOutput(global_step=11619, training_loss=0.9223433538151747, metrics={'train_runtime': 1343.412, 'train_samples_per_second': 34.591, 'train_steps_per_second': 8.649, 'total_flos': 1.0135548224077824e+17, 'train_loss': 0.9223433538151747, 'epoch': 3.0})

## ROUGE

In [58]:
import evaluate
import numpy as np

rouge = evaluate.load("rouge")

In [59]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    # If model returns tuple
    if isinstance(predictions, tuple):
        predictions = predictions[0]

    # Decode predictions
    decoded_preds = tokenizer.batch_decode(
        predictions, skip_special_tokens=True
    )

    # Replace -100 in labels (ignore index) so they can be decoded
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_labels = tokenizer.batch_decode(
        labels, skip_special_tokens=True
    )

    # Strip whitespace
    decoded_preds = [pred.strip() for pred in decoded_preds]
    decoded_labels = [label.strip() for label in decoded_labels]

    result = rouge.compute(
        predictions=decoded_preds,
        references=decoded_labels,
        use_stemmer=True
    )

    return {
        "rouge1": result["rouge1"],
        "rouge2": result["rouge2"],
        "rougeL": result["rougeL"]
    }

## Trained model ROUGE

In [ ]:
#epoch = 2

rouge = evaluate.load("rouge")

predictions = trainer.predict(validation_tokenized)

preds = predictions.predictions
labels = predictions.label_ids

labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

results = rouge.compute(
    predictions=decoded_preds,
    references=decoded_labels,
    use_stemmer=True
)

print({k: round(v * 100, 4) for k, v in results.items()})

{'rouge1': np.float64(44.6586), 'rouge2': np.float64(15.9589), 'rougeL': np.float64(24.7595), 'rougeLsum': np.float64(24.7636)}


# ROUGE results from epoch = 2

{'rouge1': np.float64(44.6586), 'rouge2': np.float64(15.9589), 'rougeL': np.float64(24.7595), 'rougeLsum': np.float64(24.7636)}


In [60]:
#epoch = 3

rouge = evaluate.load("rouge")

predictions = trainer.predict(validation_tokenized)

preds = predictions.predictions
labels = predictions.label_ids

labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

results = rouge.compute(
    predictions=decoded_preds,
    references=decoded_labels,
    use_stemmer=True
)

print({k: round(v * 100, 4) for k, v in results.items()})

{'rouge1': np.float64(44.6801), 'rouge2': np.float64(15.9898), 'rougeL': np.float64(24.713), 'rougeLsum': np.float64(24.7124)}


# ROUGE results from epoch = 3


{'rouge1': np.float64(44.6801), 'rouge2': np.float64(15.9898), 'rougeL': np.float64(24.713), 'rougeLsum': np.float64(24.7124)}